# 01 - Source Discovery

Notebook de apoyo para validar visualmente el acceso inicial a las fuentes publicas del proyecto **Municipal Revenue Big Data Analytics**.

Este notebook no reemplaza los scripts de discovery. Su proposito es revisar conectividad, estado HTTP, tipo de contenido, URL final, errores y estabilidad preliminar por fuente.

No debe guardar datos reales, archivos descargados ni outputs pesados.

## Fuentes evaluadas

Este notebook evalúa páginas de dataset y recursos directos representativos identificados durante el discovery.

| Fuente | Recurso evaluado | URL candidata | Objetivo de la prueba |
|---|---|---|---|
| MEF ingresos | Página del dataset | `https://datosabiertos.mef.gob.pe/dataset/presupuesto-y-ejecucion-de-ingreso` | Validar disponibilidad de la página oficial del dataset |
| MEF ingresos | CSV anual representativo | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/2012-Ingreso.csv` | Validar acceso directo a un archivo anual de ingresos |
| MEF ingresos | CSV diario reciente | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/2026-Ingreso-Diario.csv` | Validar existencia de recurso reciente con granularidad diaria |
| MEF ingresos | Diccionario CSV | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/Ingresos_Diccionario.csv` | Validar acceso al diccionario de datos de ingresos |
| Meta predial | Página del dataset | `https://datosabiertos.mef.gob.pe/dataset/seguimiento-de-la-meta-del-impuesto-predial` | Validar disponibilidad de la página oficial del dataset |
| Meta predial | CSV temático | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_estadistica.csv` | Validar acceso directo a tabla temática predial |
| Meta predial | CSV temático | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_preguntas.csv` | Validar acceso directo a tabla temática predial |
| Meta predial | CSV temático | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_formulario.csv` | Validar acceso directo a tabla temática predial |
| Meta predial | Diccionario CSV | `https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_respuestas_diccionario.csv` | Validar acceso directo a diccionario predial |
| RENAMU 2022 | Página del dataset | `https://www.datosabiertos.gob.pe/dataset/registro-nacional-de-municipalidades-renamu-2022-instituto-nacional-de-estadistica-e` | Validar comportamiento de la página oficial del dataset |
| RENAMU 2022 | ZIP completo | `https://www.inei.gob.pe/media/DATOS_ABIERTOS/RENAMU/DATA/2022.zip` | Validar acceso al recurso principal de ingesta |
| RENAMU 2022 | Diccionario PDF | `https://www.inei.gob.pe/media/DATOS_ABIERTOS/RENAMU/DICCIONARIO/2022/Diccionario.pdf` | Validar acceso al diccionario oficial |
| RENAMU 2022 | Muestra XLSX | `https://www.datosabiertos.gob.pe/sites/default/files/BD_Muestra_2022_0.xlsx` | Registrar comportamiento del recurso observado, aunque no sea prioritario |

Los resultados formales del discovery se documentan en `docs/source_discovery.md`.

## Preparacion del entorno

Esta celda asegura que el notebook pueda importar modulos desde `src/` aunque Jupyter se ejecute desde la carpeta `notebooks/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

## Carga de funciones de discovery

Se cargan las funciones `probe_url` implementadas en los scripts de discovery del proyecto.

In [ ]:
from dataclasses import asdict
from time import perf_counter, sleep

import pandas as pd

from src.ingestion.download_mef_income import probe_url as probe_mef_income
from src.ingestion.download_predial_goal import probe_url as probe_predial_goal
from src.ingestion.download_renamu import probe_url as probe_renamu

print('Modulos de discovery cargados correctamente')

## Configuracion de fuentes candidatas

Cada fuente se evalua con la misma cantidad de intentos para evitar sesgos. Esta prueba no representa la politica final de reintentos del pipeline; solo sirve para discovery y observacion de estabilidad.

In [ ]:
SOURCES_TO_PROBE = [
    {
        "source_name": "mef_income_dataset_page",
        "display_name": "MEF - Pagina dataset ingresos",
        "url": "https://datosabiertos.mef.gob.pe/dataset/presupuesto-y-ejecucion-de-ingreso",
        "probe_function": probe_mef_income,
    },
    {
        "source_name": "mef_income_annual_2012",
        "display_name": "MEF - CSV anual 2012",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/2012-Ingreso.csv",
        "probe_function": probe_mef_income,
    },
    {
        "source_name": "mef_income_daily_2026",
        "display_name": "MEF - CSV diario 2026",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/2026-Ingreso-Diario.csv",
        "probe_function": probe_mef_income,
    },
    {
        "source_name": "mef_income_dictionary",
        "display_name": "MEF - Diccionario ingresos",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/Ingresos_Diccionario.csv",
        "probe_function": probe_mef_income,
    },
    {
        "source_name": "predial_dataset_page",
        "display_name": "Predial - Pagina dataset",
        "url": "https://datosabiertos.mef.gob.pe/dataset/seguimiento-de-la-meta-del-impuesto-predial",
        "probe_function": probe_predial_goal,
    },
    {
        "source_name": "predial_estadistica",
        "display_name": "Predial - rentas_estadistica.csv",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_estadistica.csv",
        "probe_function": probe_predial_goal,
    },
    {
        "source_name": "predial_preguntas",
        "display_name": "Predial - rentas_preguntas.csv",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_preguntas.csv",
        "probe_function": probe_predial_goal,
    },
    {
        "source_name": "predial_formulario",
        "display_name": "Predial - rentas_formulario.csv",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_formulario.csv",
        "probe_function": probe_predial_goal,
    },
    {
        "source_name": "predial_respuestas_dictionary",
        "display_name": "Predial - diccionario respuestas",
        "url": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/rentas_respuestas_diccionario.csv",
        "probe_function": probe_predial_goal,
    },
    {
        "source_name": "renamu_dataset_page",
        "display_name": "RENAMU - Pagina dataset",
        "url": "https://www.datosabiertos.gob.pe/dataset/registro-nacional-de-municipalidades-renamu-2022-instituto-nacional-de-estadistica-e",
        "probe_function": probe_renamu,
    },
    {
        "source_name": "renamu_full_zip",
        "display_name": "RENAMU - ZIP completo",
        "url": "https://www.inei.gob.pe/media/DATOS_ABIERTOS/RENAMU/DATA/2022.zip",
        "probe_function": probe_renamu,
    },
    {
        "source_name": "renamu_dictionary_pdf",
        "display_name": "RENAMU - Diccionario PDF",
        "url": "https://www.inei.gob.pe/media/DATOS_ABIERTOS/RENAMU/DICCIONARIO/2022/Diccionario.pdf",
        "probe_function": probe_renamu,
    },
    {
        "source_name": "renamu_sample_xlsx",
        "display_name": "RENAMU - Muestra XLSX",
        "url": "https://www.datosabiertos.gob.pe/sites/default/files/BD_Muestra_2022_0.xlsx",
        "probe_function": probe_renamu,
    },
]

ATTEMPTS = 1
TIMEOUT_SECONDS = 30
WAIT_SECONDS = 3

pd.DataFrame(
    [
        {
            "source_name": source["source_name"],
            "display_name": source["display_name"],
            "url": source["url"],
            "attempts": ATTEMPTS,
            "timeout_seconds": TIMEOUT_SECONDS,
        }
        for source in SOURCES_TO_PROBE
    ]
)

## Funcion de multiples intentos

Esta funcion ejecuta multiples intentos por fuente y devuelve un resultado tabular con:

- fuente;
- intento;
- estado HTTP;
- tipo de contenido;
- URL final;
- duracion;
- error, si aplica;
- clasificacion simple del resultado.

Esto permite comparar estabilidad entre fuentes.

In [ ]:
def classify_probe_result(status_code, content_type, error):
    if error:
        if 'Timeout' in error:
            return 'TIMEOUT'
        return 'ERROR'

    if status_code is None:
        return 'NO_STATUS'

    if 200 <= status_code < 300:
        if content_type and 'text/html' in content_type.lower():
            return 'HTML_OK'
        return 'OK'

    if 300 <= status_code < 400:
        return 'REDIRECT'

    if status_code in {403, 404, 405, 418, 429}:
        return 'SPECIAL_HTTP_RESPONSE'

    if status_code >= 500:
        return 'SERVER_ERROR'

    return 'OTHER_HTTP_RESPONSE'


def run_probe_attempts_for_source(
    source_name,
    display_name,
    url,
    probe_function,
    attempts=3,
    timeout_seconds=30,
    wait_seconds=5,
):
    rows = []

    for attempt_number in range(1, attempts + 1):
        print(f'[{source_name}] intento {attempt_number}/{attempts}')

        started = perf_counter()

        result = probe_function(
            source_name=source_name,
            url=url,
            timeout_seconds=timeout_seconds,
        )

        duration_seconds = round(perf_counter() - started, 2)
        row = asdict(result)

        row['display_name'] = display_name
        row['attempt_number'] = attempt_number
        row['duration_seconds'] = duration_seconds
        row['result_category'] = classify_probe_result(
            status_code=row.get('status_code'),
            content_type=row.get('content_type'),
            error=row.get('error'),
        )

        rows.append(row)

        print(
            f"  status={row.get('status_code')} | "
            f"category={row.get('result_category')} | "
            f"duration={duration_seconds}s"
        )

        if row.get('error'):
            print(f"  error={row.get('error')}")

        if attempt_number < attempts:
            sleep(wait_seconds)

    return rows


def run_all_source_probes(
    sources,
    attempts=3,
    timeout_seconds=30,
    wait_seconds=5,
):
    all_rows = []

    for source in sources:
        print('=' * 80)
        print(source['display_name'])

        source_rows = run_probe_attempts_for_source(
            source_name=source['source_name'],
            display_name=source['display_name'],
            url=source['url'],
            probe_function=source['probe_function'],
            attempts=attempts,
            timeout_seconds=timeout_seconds,
            wait_seconds=wait_seconds,
        )

        all_rows.extend(source_rows)

    return pd.DataFrame(all_rows)

## Ejecucion de discovery con multiples intentos

Esta celda ejecuta la misma prueba sobre todas las fuentes candidatas.

In [ ]:
discovery_attempts_df = run_all_source_probes(
    sources=SOURCES_TO_PROBE,
    attempts=ATTEMPTS,
    timeout_seconds=TIMEOUT_SECONDS,
    wait_seconds=WAIT_SECONDS,
)

discovery_attempts_df

## Vista resumida por intento

Esta tabla muestra solo las columnas principales para interpretar rapidamente los resultados.

In [ ]:
summary_columns = [
    'source_name',
    'attempt_number',
    'status_code',
    'content_type',
    'duration_seconds',
    'result_category',
    'final_url',
    'error',
]

available_columns = [
    column for column in summary_columns if column in discovery_attempts_df.columns
]

discovery_attempts_df[available_columns]

## Resumen por fuente

Esta tabla ayuda a ver cuantas veces respondio cada fuente y cuantas veces fallo.

In [ ]:
source_summary_df = (
    discovery_attempts_df
    .groupby(['source_name', 'result_category'], dropna=False)
    .size()
    .reset_index(name='attempt_count')
    .sort_values(['source_name', 'result_category'])
)

source_summary_df

## Interpretacion tecnica

- `HTML_OK`: la página HTML responde correctamente. Puede corresponder a una página de dataset o portal, no a un archivo descargable.
- `OK`: la fuente respondio correctamente y podria corresponder a un recurso no HTML.
- `TIMEOUT`: la fuente no respondio dentro del tiempo configurado.
- `SPECIAL_HTTP_RESPONSE`: el sitio respondio con un codigo especial como 403, 405, 418 o 429.
- `SERVER_ERROR`: el servidor respondio con error 5xx.

Una fuente que responde algunas veces y falla en otras debe considerarse intermitente. Para la ingesta definitiva se deben contemplar reintentos configurables, timeout, auditoria por intento y fallback controlado.

## Conclusion esperada del discovery

Este notebook permite comparar paginas de dataset y recursos directos representativos con un criterio uniforme.

- Si una pagina de dataset devuelve `HTML_OK`, significa que el portal responde correctamente, pero no representa todavia un archivo descargable.
- Si un recurso CSV, ZIP o PDF devuelve `OK`, se considera un candidato valido para la ingesta o documentacion.
- Si un recurso devuelve `SPECIAL_HTTP_RESPONSE`, como HTTP 418, se registra como recurso observado pero no prioritario para la ingesta.
- La ingesta definitiva no debe implementarse hasta definir rango temporal, granularidad, tablas necesarias, estrategia de descarga y auditoria.